This notebook will cover:

Training time,
Inference time,
Model parameter count,
Model/file size,
Memory/computational considerations,
Comparison of classical ML vs DL,
Identify where lightweight optimization may be useful

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import pandas as pd
import torch

project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'

data_path = project_path + '/data/raw/archive/data/data'
test_path = data_path + '/test'
models_path = project_path + '/models'
results_path = project_path + '/results'

channel = "A-8"

test_data = np.load(
    test_path + "/" + channel + ".npy"
)

test_telemetry = test_data[:, 0]

print("Project path:", project_path)
print("Channel:", channel)
print("Test telemetry shape:", test_telemetry.shape)
print("Models folder exists:", os.path.exists(models_path))

Mounted at /content/drive
Project path: /content/drive/MyDrive/Spacecraft-Anomaly-Detection
Channel: A-8
Test telemetry shape: (8375,)
Models folder exists: True


##Count LSTM Autoencoder Parameters

The number of parameters tells us how many trainable values the neural network contains.

In [2]:
import torch
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_size=1,
        hidden_size=32,
        latent_size=16
    ):
        super().__init__()

        self.encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.encoder_fc = nn.Linear(
            hidden_size,
            latent_size
        )

        self.decoder_fc = nn.Linear(
            latent_size,
            hidden_size
        )

        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            hidden_size,
            input_size
        )

    def forward(self, x):
        encoded_sequence, (hidden, cell) = self.encoder(x)

        latent = self.encoder_fc(hidden[-1])

        decoder_input = self.decoder_fc(latent)
        decoder_input = decoder_input.unsqueeze(1)
        decoder_input = decoder_input.repeat(1, x.size(1), 1)

        decoded_sequence, _ = self.decoder(decoder_input)

        output = self.output_layer(decoded_sequence)

        return output


model = LSTMAutoencoder()

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

Total parameters: 14033
Trainable parameters: 14033


##Model File Size

In [3]:
model_path = models_path + "/lstm_autoencoder_A8.pt"

model_size_bytes = os.path.getsize(model_path)
model_size_kb = model_size_bytes / 1024
model_size_mb = model_size_kb / 1024

print("Model file:", model_path)
print("Model size (bytes):", model_size_bytes)
print("Model size (KB):", round(model_size_kb, 2))
print("Model size (MB):", round(model_size_mb, 4))

Model file: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/models/lstm_autoencoder_A8.pt
Model size (bytes): 61609
Model size (KB): 60.17
Model size (MB): 0.0588


| Measure         |        Result |
| --------------- | ------------: |
| Parameters      |    **14,033** |
| Model file size |  **60.17 KB** |
| Model size      | **0.0588 MB** |

This is useful because the model is relatively compact in storage, but storage size alone doesn't tell us whether it is computationally lightweight.

##Measure LSTM Inference Time

We'll use the same A-8 test windows from SERIAL 26.

In [4]:
window_size = 50

def create_windows(telemetry, window_size=50):
    windows = []

    for i in range(len(telemetry) - window_size + 1):
        windows.append(
            telemetry[i:i + window_size]
        )

    return np.array(windows)


test_windows = create_windows(
    test_telemetry,
    window_size
)

X_test = torch.tensor(
    test_windows,
    dtype=torch.float32
).unsqueeze(-1)

model.eval()

with torch.no_grad():
    start_time = time.perf_counter()

    reconstructed = model(X_test)

    end_time = time.perf_counter()

total_inference_time = end_time - start_time

num_windows = len(X_test)

time_per_window = (
    total_inference_time / num_windows
)

print("Number of test windows:", num_windows)
print(
    "Total inference time (seconds):",
    round(total_inference_time, 6)
)
print(
    "Inference time per window (ms):",
    round(time_per_window * 1000, 6)
)


Number of test windows: 8326
Total inference time (seconds): 2.609967
Inference time per window (ms): 0.313472


| Metric               |       Result |
| -------------------- | -----------: |
| Test windows         |        8,326 |
| Total inference time |  **2.610 s** |
| Time per window      | **0.313 ms** |

This gives us a useful baseline. But one measurement can be noisy, so later we may repeat inference several times and use the average.



| Measure              |              LSTM Autoencoder |
| -------------------- | ----------------------------: |
| Parameters           |                    **14,033** |
| Model size           |                  **60.17 KB** |
| Inference time       | **2.610 s** for 8,326 windows |
| Per-window inference |                  **0.313 ms** |




##Measure LSTM Training Time

We will train the same architecture using the same A-8 training windows and measure how long training takes.

In [5]:
train_data = np.load(
    data_path + "/train/" + channel + ".npy"
)

train_telemetry = train_data[:, 0]

train_windows = create_windows(
    train_telemetry,
    window_size
)

X_train = torch.tensor(
    train_windows,
    dtype=torch.float32
).unsqueeze(-1)

print("Training windows:", X_train.shape)

Training windows: torch.Size([713, 50, 1])


##Measure LSTM Training Time

We’ll use the same basic LSTM Autoencoder setup and 50 epochs so the measurement is consistent with our earlier experiment.

In [6]:
model = LSTMAutoencoder()

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 50

start_time = time.perf_counter()

for epoch in range(epochs):
    optimizer.zero_grad()

    reconstructed = model(X_train)

    loss = criterion(
        reconstructed,
        X_train
    )

    loss.backward()
    optimizer.step()

end_time = time.perf_counter()

training_time = end_time - start_time
average_epoch_time = training_time / epochs

print("Training epochs:", epochs)
print("Total training time (seconds):", round(training_time, 4))
print("Average time per epoch (seconds):", round(average_epoch_time, 4))
print("Final training loss:", round(loss.item(), 6))

Training epochs: 50
Total training time (seconds): 13.1605
Average time per epoch (seconds): 0.2632
Final training loss: 0.08049


| Measurement         |        Result |
| ------------------- | ------------: |
| Epochs              |            50 |
| Total training time | **13.1605 s** |
| Average/epoch       |  **0.2632 s** |
| Final training loss |   **0.08049** |


This gives us a real computational-cost baseline for the LSTM Autoencoder on the current CPU Colab runtime

##We'll measure the computational cost of Isolation Forest, using the same A-8 telemetry data and the same configuration from our earlier experiment.

In [7]:
from sklearn.ensemble import IsolationForest

X_train_classical = train_telemetry.reshape(-1, 1)

isolation_forest = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

start_time = time.perf_counter()

isolation_forest.fit(X_train_classical)

end_time = time.perf_counter()

if_training_time = end_time - start_time

print("Training samples:", X_train_classical.shape)
print("Number of trees:", 100)
print("Isolation Forest training time (seconds):", round(if_training_time, 6))

Training samples: (762, 1)
Number of trees: 100
Isolation Forest training time (seconds): 1.28972


##Isolation Forest Inference Time

Now we'll measure how long Isolation Forest takes to score the A-8 test data.

In [8]:
X_test_classical = test_telemetry.reshape(-1, 1)

isolation_forest = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

isolation_forest.fit(X_train_classical)

start_time = time.perf_counter()

if_predictions = isolation_forest.predict(X_test_classical)

end_time = time.perf_counter()

if_inference_time = end_time - start_time
if_time_per_sample = if_inference_time / len(X_test_classical)

print("Test samples:", X_test_classical.shape)
print("Total inference time (seconds):", round(if_inference_time, 6))
print("Inference time per sample (ms):", round(if_time_per_sample * 1000, 6))

Test samples: (8375, 1)
Total inference time (seconds): 0.047933
Inference time per sample (ms): 0.005723


##We now have the first computational comparison:

| Model            | Training time | Inference time |
| ---------------- | ------------: | -------------: |
| Isolation Forest | **1.28972 s** | **0.047933 s** |
| LSTM Autoencoder | **13.1605 s** | **2.609967 s** |


These are A-8 CPU benchmark measurements, so we'll keep the hardware and dataset context with them.

##Autoencoder Parameter Count

Next, we'll measure the size of the basic Autoencoder we used in SERIAL 20.

In [9]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(1, 4),
            nn.ReLU(),
            nn.Linear(4, 2)
        )

        self.decoder = nn.Sequential(
            nn.Linear(2, 4),
            nn.ReLU(),
            nn.Linear(4, 1)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


autoencoder = Autoencoder()

ae_parameters = sum(
    parameter.numel()
    for parameter in autoencoder.parameters()
)

ae_trainable_parameters = sum(
    parameter.numel()
    for parameter in autoencoder.parameters()
    if parameter.requires_grad
)

print("Autoencoder total parameters:", ae_parameters)
print("Autoencoder trainable parameters:", ae_trainable_parameters)

Autoencoder total parameters: 35
Autoencoder trainable parameters: 35


###Autoencoder Model Size

Now let's measure the saved Autoencoder model file size.

In [10]:
ae_model_path = models_path + "/autoencoder_A8.pt"

if os.path.exists(ae_model_path):
    ae_model_size_bytes = os.path.getsize(ae_model_path)
    ae_model_size_kb = ae_model_size_bytes / 1024
    ae_model_size_mb = ae_model_size_kb / 1024

    print("Autoencoder model file:", ae_model_path)
    print("Model size (bytes):", ae_model_size_bytes)
    print("Model size (KB):", round(ae_model_size_kb, 2))
    print("Model size (MB):", round(ae_model_size_mb, 4))
else:
    print("Autoencoder model file not found:", ae_model_path)

Autoencoder model file not found: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/models/autoencoder_A8.pt


##Measure Autoencoder Training Time

We'll train the same small Autoencoder for 50 epochs on the A-8 training data and measure the time.

In [11]:
ae_model = Autoencoder()

ae_criterion = torch.nn.MSELoss()
ae_optimizer = torch.optim.Adam(
    ae_model.parameters(),
    lr=0.001
)

X_train_ae = torch.tensor(
    train_telemetry,
    dtype=torch.float32
).unsqueeze(1)

epochs = 50

start_time = time.perf_counter()

for epoch in range(epochs):
    ae_optimizer.zero_grad()

    reconstructed = ae_model(X_train_ae)

    loss = ae_criterion(
        reconstructed,
        X_train_ae
    )

    loss.backward()
    ae_optimizer.step()

end_time = time.perf_counter()

ae_training_time = end_time - start_time
ae_average_epoch_time = ae_training_time / epochs

print("Training samples:", X_train_ae.shape)
print("Training epochs:", epochs)
print("Total Autoencoder training time (seconds):", round(ae_training_time, 6))
print("Average time per epoch (seconds):", round(ae_average_epoch_time, 6))
print("Final training loss:", round(loss.item(), 6))

Training samples: torch.Size([762, 1])
Training epochs: 50
Total Autoencoder training time (seconds): 0.288594
Average time per epoch (seconds): 0.005772
Final training loss: 1.014185


###Autoencoder Inference Time

Now we'll measure how quickly the Autoencoder processes the A-8 test telemetry.

In [12]:
X_test_ae = torch.tensor(
    test_telemetry,
    dtype=torch.float32
).unsqueeze(1)

ae_model.eval()

with torch.no_grad():
    start_time = time.perf_counter()

    reconstructed_ae = ae_model(X_test_ae)

    end_time = time.perf_counter()

ae_inference_time = end_time - start_time
ae_time_per_sample = ae_inference_time / len(X_test_ae)

print("Test samples:", X_test_ae.shape)
print("Total Autoencoder inference time (seconds):", round(ae_inference_time, 6))
print("Inference time per sample (ms):", round(ae_time_per_sample * 1000, 6))

Test samples: torch.Size([8375, 1])
Total Autoencoder inference time (seconds): 0.00131
Inference time per sample (ms): 0.000156


#One-Class SVM Training Time

Now let's measure the other classical ML model from our experiments.

We'll use the same configuration as SERIAL 18/25:

In [13]:
from sklearn.svm import OneClassSVM

one_class_svm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

start_time = time.perf_counter()

one_class_svm.fit(X_train_classical)

end_time = time.perf_counter()

ocsvm_training_time = end_time - start_time

print("Training samples:", X_train_classical.shape)
print("Kernel:", "RBF")
print("nu:", 0.05)
print("One-Class SVM training time (seconds):", round(ocsvm_training_time, 6))

Training samples: (762, 1)
Kernel: RBF
nu: 0.05
One-Class SVM training time (seconds): 0.011314


##One-Class SVM Inference Time

In [14]:
one_class_svm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

one_class_svm.fit(X_train_classical)

start_time = time.perf_counter()

ocsvm_predictions = one_class_svm.predict(X_test_classical)

end_time = time.perf_counter()

ocsvm_inference_time = end_time - start_time
ocsvm_time_per_sample = ocsvm_inference_time / len(X_test_classical)

print("Test samples:", X_test_classical.shape)
print("Total One-Class SVM inference time (seconds):", round(ocsvm_inference_time, 6))
print("Inference time per sample (ms):", round(ocsvm_time_per_sample * 1000, 6))

Test samples: (8375, 1)
Total One-Class SVM inference time (seconds): 0.038823
Inference time per sample (ms): 0.004636


##Measure Statistical Processing Time

For Global Z-score, there isn't really a model-training stage. We'll measure its anomaly-detection processing time instead

In [15]:
start_time = time.perf_counter()

train_mean = np.mean(train_telemetry)
train_std = np.std(train_telemetry)

test_z_scores = (
    (test_telemetry - train_mean)
    / train_std
)

global_z_predictions = (
    np.abs(test_z_scores) > 3
).astype(int)

end_time = time.perf_counter()

global_z_time = end_time - start_time
global_z_time_per_sample = global_z_time / len(test_telemetry)

print("Test samples:", len(test_telemetry))
print("Global Z-score processing time (seconds):", round(global_z_time, 6))
print(
    "Processing time per sample (ms):",
    round(global_z_time_per_sample * 1000, 6)
)

Test samples: 8375
Global Z-score processing time (seconds): 0.016078
Processing time per sample (ms): 0.00192


##Rolling Z-score Processing Time

Now we'll measure the other statistical method using the same window size = 50 and past-only calculation we used in SERIAL 14.

In [16]:
window_size = 50

start_time = time.perf_counter()

test_series = pd.Series(test_telemetry)

rolling_mean = (
    test_series
    .shift(1)
    .rolling(
        window=window_size,
        min_periods=window_size
    )
    .mean()
)

rolling_std = (
    test_series
    .shift(1)
    .rolling(
        window=window_size,
        min_periods=window_size
    )
    .std()
)

rolling_z_scores = (
    (test_series - rolling_mean)
    / rolling_std
)

rolling_predictions = (
    np.abs(rolling_z_scores) > 3
).astype(int)

end_time = time.perf_counter()

rolling_z_time = end_time - start_time
rolling_z_time_per_sample = rolling_z_time / len(test_telemetry)

print("Test samples:", len(test_telemetry))
print("Window size:", window_size)
print(
    "Rolling Z-score processing time (seconds):",
    round(rolling_z_time, 6)
)
print(
    "Processing time per sample (ms):",
    round(rolling_z_time_per_sample * 1000, 6)
)

Test samples: 8375
Window size: 50
Rolling Z-score processing time (seconds): 0.017418
Processing time per sample (ms): 0.00208


Create Computational Cost Summary

Now let's save everything we've measured into a CSV so the results become part of the research project
.

In [17]:
computational_cost = pd.DataFrame([
    {
        "method": "Global Z-score",
        "parameters": np.nan,
        "training_or_processing_time_seconds": global_z_time,
        "inference_time_seconds": np.nan,
        "input_type": "single telemetry point",
        "notes": "Processing time; no model training"
    },
    {
        "method": "Rolling Z-score",
        "parameters": np.nan,
        "training_or_processing_time_seconds": rolling_z_time,
        "inference_time_seconds": np.nan,
        "input_type": "single telemetry point with rolling window",
        "notes": "Processing time; window size 50; past-only"
    },
    {
        "method": "Isolation Forest",
        "parameters": np.nan,
        "training_or_processing_time_seconds": if_training_time,
        "inference_time_seconds": if_inference_time,
        "input_type": "single telemetry point",
        "notes": "100 trees; contamination='auto'"
    },
    {
        "method": "One-Class SVM",
        "parameters": np.nan,
        "training_or_processing_time_seconds": ocsvm_training_time,
        "inference_time_seconds": ocsvm_inference_time,
        "input_type": "single telemetry point",
        "notes": "RBF kernel; nu=0.05"
    },
    {
        "method": "Autoencoder",
        "parameters": ae_parameters,
        "training_or_processing_time_seconds": ae_training_time,
        "inference_time_seconds": ae_inference_time,
        "input_type": "single telemetry point",
        "notes": "35 trainable parameters"
    },
    {
        "method": "LSTM Autoencoder",
        "parameters": total_parameters,
        "training_or_processing_time_seconds": training_time,
        "inference_time_seconds": total_inference_time,
        "input_type": "50-timestep sequence",
        "notes": "14,033 trainable parameters"
    }
])

cost_path = results_path + "/computational_cost_A8.csv"

computational_cost.to_csv(
    cost_path,
    index=False
)

print("Saved:", cost_path)
print()
print(computational_cost)

Saved: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/results/computational_cost_A8.csv

             method  parameters  training_or_processing_time_seconds  \
0    Global Z-score         NaN                             0.016078   
1   Rolling Z-score         NaN                             0.017418   
2  Isolation Forest         NaN                             1.289720   
3     One-Class SVM         NaN                             0.011314   
4       Autoencoder        35.0                             0.288594   
5  LSTM Autoencoder     14033.0                            13.160519   

   inference_time_seconds                                  input_type  \
0                     NaN                      single telemetry point   
1                     NaN  single telemetry point with rolling window   
2                0.047933                      single telemetry point   
3                0.038823                      single telemetry point   
4                0.001310           

his gives us evidence for your lightweight research direction:

The computational-cost experiment shows that the LSTM Autoencoder provides temporal modeling capability at a substantially higher computational cost than the small feed-forward Autoencoder and classical methods under the evaluated A-8 CPU configuration.

#github commit


In [ ]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git status --short

!git add notebooks/27_computational_cost_analysis.ipynb
!git add results/computational_cost_A8.csv

!git commit -m "Complete SERIAL 27 computational cost analysis"

!git push origin main

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Refresh index: 100% (50/50), done.
 M notebooks/13_evaluate_global_zscore.ipynb
 M notebooks/14_rolling_zscore.ipynb
 M notebooks/15_compare_statistical_methods.ipynb
 M notebooks/16_isolation_forest.ipynb
 M notebooks/17_evaluate_isolation_forest.ipynb
 M notebooks/18_one_class_svm.ipynb
 M notebooks/19_compare_classical_methods.ipynb
 M notebooks/20_autoencoder.ipynb
 M notebooks/21_analyze_autoencoder.ipynb
 M notebooks/22_time_series_windows.ipynb
 M notebooks/23_lstm_autoencoder.ipynb
 M notebooks/24_evaluate_lstm_autoencoder.ipynb
 M notebooks/25_model_comparison_and_research_gap.ipynb
 M notebooks/26_explainable_ai.ipynb
?? notebooks/27_computational_cost_analysis.ipynb
?? results/computational_cost_A8.csv
